In [ ]:
# Instalacja pakietów
!pip install ultralytics

In [ ]:
# Załadowanie bibliotek
import torch, torch.nn.functional as F
import os, cv2, numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm
from ultralytics import YOLO

In [ ]:
# Ustawienie katalogu roboczego
from google.colab import drive
drive.mount('/content/drive')
os.chdir('/content/drive/My Drive/Colab Notebooks/')

In [ ]:
# Załadowanie modelu
model = YOLO('yolov11n-cls.pt')
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.model.to(device)

In [ ]:
# Funkcja zwracająca prawdopodobieństwo klasy
def predict_proba(img_tensor):
    results = model(img_tensor)
    probs = results[0].probs.data
    return probs

In [ ]:
# Wczytanie obrazu
def load_image(path, img_size = 640):
    img = cv2.imread(path)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    img = cv2.resize(img, (img_size, img_size))
    img = img.astype(np.float32) / 255.0
    img_tensor = torch.tensor(img).permute(2, 0, 1).unsqueeze(0)
    return img, img_tensor.to(device)

In [ ]:
# Funkcja generująca maski
def generate_masks(N, s, p1, input_size):
    cell_size = np.ceil(np.array(input_size) / s)
    up_size = (s + 1) * cell_size
    masks = []
    for _ in range(N):
        grid = np.random.rand(s, s) < p1
        grid = grid.astype(np.float32)
        mask = cv2.resize(grid,
            (int(up_size[1]), int(up_size[0])),
            interpolation = cv2.INTER_LINEAR)
        x = np.random.randint(0, cell_size[0])
        y = np.random.randint(0, cell_size[1])
        mask = mask[x:x+input_size[0], y:y+input_size[1]]
        masks.append(mask)
    masks = np.array(masks)
    return masks

In [ ]:
# Generowanie mapy istotności RISE
def rise_explanation(img_tensor, target_class, N, s, p1):
    _, _, H, W = img_tensor.shape
    masks = generate_masks(N, s, p1, (H, W))
    masks_tensor = torch.tensor(masks).unsqueeze(1).to(device)
    saliency = torch.zeros((H, W)).to(device)
    for i in tqdm(range(N)):
        mask = masks_tensor[i]
        masked_img = img_tensor * mask
        probs = predict_proba(masked_img)
        score = probs[target_class]
        saliency += score * mask.squeeze(0)
    saliency = saliency / (N * p1)
    saliency = saliency.detach().cpu().numpy()
    return saliency

In [ ]:
# Wybór klasy do wyjaśnienia
image_path = 'image.jpg' # Należy podać nazwę pliku obrazu
original_img, img_tensor = load_image(image_path)
with torch.no_grad():
    probs = predict_proba(img_tensor)
    predicted_class = torch.argmax(probs).item()

In [ ]:
# Wywołanie funkcji rise_explanation()
saliency_map = rise_explanation(
    img_tensor,
    target_class = predicted_class,
    N = 2000,
    s = 8,
    p1 = 0.1
)

In [ ]:
# Wizualizacja mapy istotności
def show_saliency(original_img, saliency):
    saliency = (saliency - saliency.min()) / (saliency.max() - saliency.min())
    saliency = cv2.resize(saliency, (original_img.shape[1], original_img.shape[0]))
    heatmap = cv2.applyColorMap(np.uint8(255 * saliency), cv2.COLORMAP_JET)
    heatmap = cv2.cvtColor(heatmap, cv2.COLOR_BGR2RGB)
    overlay = 0.5 * original_img + 0.5 * heatmap / 255.0
    overlay = np.clip(overlay, 0, 1)
    plt.imshow(overlay)
    class_names = list(model.names.values())
    plt.title('Predykcja: ' + str(class_names[predicted_class]))
    plt.axis('off')
    plt.show()

In [ ]:
# Uruchomienie kodu
show_saliency(original_img, saliency_map)